### Requirements 

#### .env file

```bash
MILVUS_URL = # your milvus server url
MILVUS_TOKEN = # your milvus access token
MILVUS_COLLECTION_NAME = # milvus collection name
```

In [1]:
import os
import sys 
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(override=True)
ROOT = Path.cwd().parent.parent.parent.resolve()

print(f"ROOT: {ROOT}")
sys.path.append(str(ROOT))

ROOT: /users/oshan/Dev/financial-document-based-agent-system


### Milvus server Info

In [2]:
import pymilvus
import os
from urllib.parse import urlparse

In [3]:
milvus_url = os.environ.get("MILVUS_URL")
milvus_token = os.environ.get("MILVUS_TOKEN")
milvus_collection_name = os.environ.get("MILVUS_COLLECTION_NAME")

if not milvus_url:
    raise RuntimeError("MILVUS_URL environment variable not set")

# parse host/port if needed
u = urlparse(milvus_url if "://" in milvus_url else f"//{milvus_url}")
host = u.hostname or milvus_url
port = u.port or 19530

# try connecting (try uri first, then host/port with optional token as password)
errors = []
try:
    pymilvus.connections.connect(uri=milvus_url)
except Exception as e:
    errors.append(e)
    try:
        if milvus_token:
            pymilvus.connections.connect(host=host, port=str(port), password=milvus_token)
        else:
            pymilvus.connections.connect(host=host, port=str(port))
    except Exception as e2:
        errors.append(e2)
        raise RuntimeError("Failed to connect to Milvus", errors)

print("Connected to:", milvus_url)
print("Collections:", pymilvus.utility.list_collections())

Connected to: http://localhost:19530
Collections: ['xml_documents', 'xml_documents_e5', 'xml_documents_qwen8', 'e2e_1st', 'my_rag_collection', 'xml_documentsv3', 'xml_documentsv4', 'financial_documents', 'financial_documents_test', 'xml_documents_bge', 'e2e_2st', 'embedchain_store', 'xml_documentsv5', 'xml_documents_nomic']


In [5]:
milvus_url = os.environ.get("MILVUS_URL")
milvus_token = os.environ.get("MILVUS_TOKEN")
milvus_collection_name = os.environ.get("MILVUS_COLLECTION_NAME")

In [6]:
milvus_collection_name in pymilvus.utility.list_collections()

True

### Class info

In [7]:
from cgcore.vectordb.milvus import MilvusDB
from cgcore.configs.vectordb.milvus import MilvusConfig

In [8]:
config = MilvusConfig(**{
    "collection_name": milvus_collection_name,
    "dimensions": 1536,
})

milvus_db = MilvusDB(config)

MilvusClient connected.
pymilvus ORM connected to localhost:19530 for setup.
Collection 'financial_documents_test' already exists. Skipping creation.


In [9]:
config.collection_name

'financial_documents_test'

#### Insert text

In [8]:
question = "What is Milvus?"
embedding = [0.0] * 1536  # Dummy embedding for testing
content = "Milvus is an open-source vector database."
docid = "test_paper_1"

milvus_db.insert([
    {
        "content": content, 
        "vector": embedding, 
        "doc_id": docid, 
        "meta_data": {"source": "test_source"}
    },
    {
        "content": "Milvus supports efficient similarity search.", 
        "vector": embedding, 
        "doc_id": "test_paper_2", 
        "meta_data": {"source": "test_source"}
    }
])

True

#### Retrived text

In [9]:
milvus_db.vector_search(embedding, top_k=2)

[DEBUG] Returning 0 formatted results


[]

### Drop collection if necessary

In [10]:
if pymilvus.utility.has_collection(config.collection_name):
    pymilvus.utility.drop_collection(config.collection_name)
    print(f"Dropped collection: {config.collection_name}")
else:
    print(f"Collection not found: {config.collection_name}")
print("Collections now:", pymilvus.utility.list_collections())

Dropped collection: financial_documents_test
Collections now: ['xml_documentsv3', 'xml_documentsv4', 'financial_documents', 'xml_documents_bge', 'e2e_2st', 'embedchain_store', 'xml_documentsv5', 'xml_documents_nomic', 'xml_documents', 'xml_documents_e5', 'xml_documents_qwen8', 'e2e_1st', 'my_rag_collection']
